# Constraint Programming with OR-Tools CP-SAT

In this notebook, you will get hands-on experience with the powerful OR-Tools CP-SAT solver. By the end, you should be able to:

* **Create** a Constraint Programming (CP) model
* **Define** integer, boolean, and interval variables
* **Implement** fundamental constraints, including logical conditions (`OnlyEnforceIf`)
* **Understand and use** scheduling-specific global constraints like `NoOverlap`

**Reference:** [OR-Tools CP-SAT Documentation](https://developers.google.com/optimization/cp/cp_solver)

## 1. OR-Tools Basics

In [ ]:
from ortools.sat.python import cp_model

### 1.1 Creating a CP Model

In [ ]:
model = cp_model.CpModel()

### 1.2 Creating Integer Variables

In [ ]:
# Explore the documentation
model.NewIntVar??

In [ ]:
x = model.NewIntVar(0, 10, "x")
y = model.NewIntVar(0, 20, "y")

### 1.3 Creating Constraints

In [ ]:
model.Add(x + y <= 10)

### 1.4 Setting Objective Function

In [ ]:
model.Maximize(x + y)

### 1.5 Solving

In [ ]:
def solve(model, enumerate_all: bool = False):
    solver = cp_model.CpSolver()
    solver.parameters.enumerate_all_solutions = enumerate_all
    solver.parameters.max_time_in_seconds = 10
    
    status = solver.Solve(model)
    status_human = solver.StatusName(status)
    print(status_human)
    return solver

In [ ]:
solver = solve(model)
value_x = solver.Value(x)
value_y = solver.Value(y)
print("Value x:", value_x, "\nValue y:", value_y)

## 2. Boolean Variables

Boolean variables {0, 1} are a special case of integer variables. The CP**SAT** solver is particularly efficient with boolean (SAT) problems.

In [ ]:
def testing_model():
    model = cp_model.CpModel()
    x = model.NewBoolVar("x")
    y = model.NewBoolVar("y")
    return model, x, y

### 2.1 Boolean Constraints

#### Implications

Constraint $x \rightarrow y$: if x is 1, then y must be 1.

In [ ]:
model, x, y = testing_model()
model.AddImplication(x, y)

### 2.2 Boolean Operators: And, Or, Xor

In [ ]:
# AND: Both must be true
model, x, y = testing_model()
model.AddBoolAnd(x, y)
solver = solve(model)
print("AND - x:", solver.Value(x), "y:", solver.Value(y))

In [ ]:
# OR: At least one must be true
model, x, y = testing_model()
model.AddBoolOr(x, y)
model.Maximize(x + y)
solver = solve(model)
print("OR - x:", solver.Value(x), "y:", solver.Value(y))

In [ ]:
# XOR: Exactly one must be true
model, x, y = testing_model()
model.AddBoolXOr(x, y)
model.Maximize(x + y)
solver = solve(model)
print("XOR - x:", solver.Value(x), "y:", solver.Value(y))

### 2.3 Advanced Boolean Constraints

CP-SAT provides convenient constraints for common patterns:

In [ ]:
# Explore the documentation
# model.AddAtLeastOne?
# model.AddAtMostOne?
model.AddExactlyOne?

In [ ]:
import pandas as pd

def model_many_variables():
    model = cp_model.CpModel()
    df = pd.Series(range(10))
    x = model.NewBoolVarSeries("x", df.index)
    return model, x

In [ ]:
model, x = model_many_variables()
model.AddExactlyOne(x)  # Exactly one variable must be true
model.Maximize(sum(x))
solver = solve(model)
print("Selected:", solver.BooleanValues(x))

## 3. Enforcement Constraints (OnlyEnforceIf)

Conditional constraints are crucial for complex modeling. A constraint is only enforced when a boolean condition is true.

**Syntax:** `model.Add(Constraint).OnlyEnforceIf(boolean_var)`

If `boolean_var` is True → Constraint must be satisfied  
If `boolean_var` is False → Constraint is ignored

### Example: If-Then-Else Logic

**Goal:** "If x < 5, set y to 0. Otherwise, set y to 10-x"

**Implementation:**
- Create boolean `b`: true if x >= 5
- If `b` is true: y == 10 - x
- If `b` is false: y == 0

In [ ]:
model = cp_model.CpModel()
x = model.NewIntVar(0, 10, "x")
y = model.NewIntVar(0, 10, "y")
b = model.NewBoolVar("b")

# Define when b is true/false
model.Add(x <= 5).OnlyEnforceIf(b.Not())  # b is false when x <= 5
model.Add(x > 5).OnlyEnforceIf(b)         # b is true when x > 5

# Define y based on b
model.Add(y == 10 - x).OnlyEnforceIf(b)      # If x > 5
model.Add(y == 0).OnlyEnforceIf(b.Not())     # If x <= 5

# Solution callback to print all solutions
class VarArraySolutionPrinter(cp_model.CpSolverSolutionCallback):
    def __init__(self, variables: list):
        cp_model.CpSolverSolutionCallback.__init__(self)
        self.__variables = variables

    def on_solution_callback(self):
        for v in self.__variables:
            print(f"{v}={self.Value(v)}", end=" ")
        print()

solver = cp_model.CpSolver()
solver.parameters.enumerate_all_solutions = True
status = solver.Solve(model, VarArraySolutionPrinter([x, y, b]))

### 📝 Exercise: Boolean Constraint Practice

**Problem:** Encode the following:

- X is a boolean array of size 20
- Y is a boolean array of size 19
- Z is an integer variable

**Constraints:**
1. For i ∈ [0, 18]: if X[i] == X[i+1] then Y[i] must be 1
2. For i ∈ [0, 16]: Y[i] != Y[i+2]
3. Z = sum(X[::3]) + sum(Y)

**Goal:** Maximize Z

In [ ]:
def exercise_model(N=20):
    model = cp_model.CpModel()
    df = pd.Series(range(N))
    
    # YOUR CODE HERE
    # Create X, Y, Z variables
    # Add the constraints
    # Set objective
    
    return model, X, Y, Z

In [ ]:
# Solution checker (run after implementing)
def solution_checker(X, Y, Z):
    N = len(X)
    for i in range(N-1):
        if X[i] == X[i+1]:
            assert Y[i], f"Y[{i}] should be 1 when X[{i}]==X[{i+1}]"
        if i < N-3:
            assert Y[i] != Y[i+2], f"Y[{i}] should differ from Y[{i+2}]"
    assert Z == sum(X[::3]) + sum(Y), "Z should equal sum(X[::3])+sum(Y)"
    print("✓ All constraints satisfied!")

# Uncomment to test:
# model, X, Y, Z = exercise_model(N=20)
# solver = solve(model)
# X_v, Y_v, Z_v = (solver.Values(X), solver.Values(Y), solver.Value(Z))
# print("X:", X_v)
# print("Y:", Y_v)
# print("Objective value Z:", Z_v)
# solution_checker(X_v, Y_v, Z_v)

## 4. Interval Variables for Scheduling

CP-SAT has powerful support for scheduling problems through **interval variables**.

In [ ]:
model.NewIntervalVar??

### Example: Schedule a Single Task

Schedule a task with duration 10 in time window [0, 20], as late as possible.

In [ ]:
model = cp_model.CpModel()
start = model.NewIntVar(0, 20, "start")
end = model.NewIntVar(0, 20, "end")
duration = 10
task_var = model.NewIntervalVar(start, duration, end, name="task")
model.Maximize(start)  # Schedule as late as possible

solver = cp_model.CpSolver()
status = solver.Solve(model)
print("Status:", solver.StatusName(status))
print(f"Task scheduled: [{solver.Value(start)}, {solver.Value(end)}]")

## 5. Global Constraints on Intervals

### 5.1 NoOverlap Constraint

The **NoOverlap** constraint ensures that a list of intervals don't overlap in time.  
**Use case:** Tasks executed by the same machine or worker.

In [ ]:
model.AddNoOverlap?

### 5.2 Cumulative Resource Constraint

The **AddCumulative** constraint generalizes NoOverlap for resources with capacity > 1.

**Key insight:**  
`NoOverlap(tasks)` ≡ `AddCumulative(tasks, demands=[1]*n, capacity=1)`

- **NoOverlap:** Perfect for Job Shop (machine capacity = 1)
- **AddCumulative:** Essential for RCPSP (resource capacity ≥ 1)

In [ ]:
model.AddCumulative?

## 6. Optional Interval Variables

Interval variables can be marked as **optional** using a boolean flag.  
If not activated (flag=False), the interval is ignored in global constraints.

In [ ]:
def small_model_scheduling():
    model = cp_model.CpModel()
    
    # Task 1: duration 10, window [0, 20]
    start_1 = model.NewIntVar(0, 20, "start_1")
    end_1 = model.NewIntVar(0, 20, "end_1")
    is_present_1 = model.NewBoolVar("is_present_1")
    task_1 = model.NewOptionalIntervalVar(
        start=start_1, size=10, end=end_1,
        is_present=is_present_1, name="task_1"
    )
    
    # Task 2: duration 12, window [0, 20]
    start_2 = model.NewIntVar(0, 10, "start_2")
    end_2 = model.NewIntVar(0, 20, "end_2")
    is_present_2 = model.NewBoolVar("is_present_2")
    task_2 = model.NewOptionalIntervalVar(
        start=start_2, size=12, end=end_2,
        is_present=is_present_2, name="task_2"
    )
    
    # No overlap constraint
    model.AddNoOverlap([task_1, task_2])
    
    return model, start_1, end_1, is_present_1, start_2, end_2, is_present_2

In [ ]:
# Without forcing presence - tasks won't be scheduled
model, start_1, end_1, is_present_1, start_2, end_2, is_present_2 = small_model_scheduling()
solver = solve(model)
print("Task 1:", solver.Values([start_1, end_1, is_present_1]))
print("Task 2:", solver.Values([start_2, end_2, is_present_2]))

### 📝 Exercise: Force Task Presence

Modify the model to force both tasks to be present. What happens?

In [ ]:
# YOUR CODE HERE
# Hint: Use model.AddBoolAnd to force both is_present variables to be true

### Maximizing Scheduled Tasks

Instead of forcing presence, maximize the number of scheduled tasks:

In [ ]:
model, start_1, end_1, is_present_1, start_2, end_2, is_present_2 = small_model_scheduling()
model.Maximize(is_present_1 + is_present_2)
solver = solve(model)
print("Task 1:", solver.Values([start_1, end_1, is_present_1]))
print("Task 2:", solver.Values([start_2, end_2, is_present_2]))

## 7. Next Steps: Job Shop Scheduling

Now that you understand CP-SAT basics, you're ready to tackle the **Job Shop Scheduling Problem**!

### Continue with Python Modules

The Job Shop implementation has been moved to clean Python modules:

```bash
# Complete the exercises
python lesson2_cpsat/exercises.py

# Check solutions
python lesson2_cpsat/solutions.py
```

**Files:**
- `jobshop_problem.py` - Problem and solution classes
- `jobshop_utils.py` - Visualization and utilities
- `exercises.py` - Implement your CP solver
- `solutions.py` - Reference implementation

See the `README.md` for details!

## Summary

You've learned:
- ✅ Creating CP models and variables
- ✅ Boolean constraints and logic
- ✅ Enforcement constraints (OnlyEnforceIf)
- ✅ Interval variables for scheduling
- ✅ Global constraints (NoOverlap, Cumulative)
- ✅ Optional intervals

**Resources:**
- [OR-Tools CP-SAT Guide](https://developers.google.com/optimization/cp/cp_solver)
- [Scheduling Examples](https://developers.google.com/optimization/scheduling)
- [OR-Tools GitHub Examples](https://github.com/google/or-tools/tree/stable/examples/python)